### Requeriments for match folder `partidas`

Goal: Build an ETL process and make the data available in the appropriate layer

Read -> Transform -> Write (ETL)


1. Read all files in the folder 'partidas' from the data lake
2. Define the correct data schema
3. Include a column with the date when the file was ingested
4. Save the data in **Parquet** format in the appropriate layer

In [0]:
%run "./00_config_storage"

In [0]:
%run "../Modulo 4/Functions"

In [0]:
%run "../Modulo 4/Variables"

In [0]:
display(dbutils.fs.ls(path_bronze))

In [0]:
path_file_match = f"{path_bronze}/partidas/"
path_file_match_2023 = f"{path_bronze}/partidas/partidas_2023.csv"

In [0]:
df_utf8 = (
    spark.read
    .option("encoding", "UTF-8")
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(path_file_match_2023)
)

df_utf16 = (
    spark.read
    .option("encoding", "UTF-16")
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(path_file_match)
)

Databricks visualization. Run in Databricks to view.

In [0]:
df_utf8 = normalize_columns(df_utf8)
df_utf16 = normalize_columns(df_utf16)

In [0]:
from pyspark.sql.functions import col

df_utf16 = df_utf16.filter(col("league_id").isNotNull())
df_utf16 = df_utf16.select(df_utf8.columns)

display(df_utf8)
display(df_utf16)

In [0]:
df_match = df_utf8.unionByName(df_utf16)
display(df_match)


In [0]:
from pyspark.sql.functions import col

# Checking values
df_match.select("goals_home").distinct().show(50)
df_match.select("goals_away").distinct().show(50)
df_match.select("score_halftime_home").distinct().show(50)
df_match.select("score_halftime_away").distinct().show(50)
df_match.select("score_fulltime_home").distinct().show(50)

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import DateType, IntegerType, DoubleType

df_match = ( df_match
    .withColumn("date", col("date").cast(DateType()))
    .withColumn("goals_home", col("goals_home").cast(DoubleType()).cast(IntegerType()))
    .withColumn("goals_away", col("goals_away").cast(DoubleType()).cast(IntegerType()))
    .withColumn("score_halftime_home", col("score_halftime_home").cast(DoubleType()).cast(IntegerType()))
    .withColumn("score_halftime_away", col("score_halftime_away").cast(DoubleType()).cast(IntegerType()))
    .withColumn("score_fulltime_home", col("score_fulltime_home").cast(DoubleType()).cast(IntegerType()))
    .withColumn("score_fulltime_away", col("score_fulltime_away").cast(DoubleType()).cast(IntegerType()))
    )

display(df_match)
df_match.printSchema()

In [0]:
df_match_date = create_column_date(df_match)
display(df_match_date)

In [0]:
df_match_date.write.mode("overwrite").parquet(f"{path_silver}/partidas")